# Wav2Vec2-base (frozen) + SVM

**Модель:** Wav2Vec2-base (frozen)  mean+std+max  SVM RBF

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import time
import torch
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, cross_val_predict
from sklearn.metrics import make_scorer, f1_score
from transformers import Wav2Vec2Model, Wav2Vec2Processor

exp_dir = Path().resolve()
sys.path.insert(0, str(exp_dir.parent.parent))

from shared import config, data_utils, train_utils
from shared.evaluate import find_optimal_threshold, evaluate
from shared.data_utils import build_feature_matrix, load_audio
from shared.results_utils import save_result_csv

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(DEVICE)

In [ ]:
MODEL_ID = "facebook/wav2vec2-base"
w2v_proc  = Wav2Vec2Processor.from_pretrained(MODEL_ID)
w2v_model = Wav2Vec2Model.from_pretrained(MODEL_ID).to(DEVICE)
w2v_model.eval()

def extract_wav2vec(path):
    y, _ = load_audio(path, sr=16000)
    inp = w2v_proc(y, sampling_rate=16000, return_tensors="pt", padding=True)
    with torch.no_grad():
        hs = w2v_model(inp.input_values.to(DEVICE)).last_hidden_state[0].cpu().numpy()
    return np.concatenate([hs.mean(0), hs.std(0), hs.max(0)]).astype(np.float32)

In [ ]:
(
    paths_trainval, labels_trainval, letters_trainval,
    paths_test, labels_test, letters_test,
) = data_utils.get_test_split()

print(f"Train+Val: {len(paths_trainval)} good: {np.sum(labels_trainval==0)}, bad: {np.sum(labels_trainval==1)}")
print(f"Test:      {len(paths_test)}  good: {np.sum(labels_test==0)},  bad: {np.sum(labels_test==1)}")

## Подбор гиперпараметров (GridSearchCV на train+val)

In [ ]:
print("Извлечение признаков (train+val)...")
X_embeds_trainval = build_feature_matrix(paths_trainval, extract_wav2vec, n_jobs=1)
print("Извлечение признаков (test)...")
X_embeds_test     = build_feature_matrix(paths_test,     extract_wav2vec, n_jobs=1)

del w2v_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

X_trainval = np.hstack([X_embeds_trainval, letters_trainval])
X_test     = np.hstack([X_embeds_test,     letters_test])


In [ ]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(kernel="rbf", class_weight="balanced",
                probability=True, random_state=config.RANDOM_STATE)),
])
f1_bad_scorer = make_scorer(f1_score, pos_label=config.CLASS_BAD, average="binary")
param_grid = {
    "clf__C":     [0.1, 0.5, 1.0, 2.0, 5.0, 10.0, 50.0],
    "clf__gamma": ["scale", "auto", 0.001, 0.01, 0.05, 0.1],
}

t0 = time.perf_counter()
grid = GridSearchCV(pipeline, param_grid, cv=5,
                    scoring=f1_bad_scorer, refit=True, n_jobs=-1)
grid.fit(X_trainval, labels_trainval)
train_time_sec = time.perf_counter() - t0

best_params = grid.best_params_
best_clf = grid.best_estimator_
print(f"Лучшие параметры: {best_params}")

## Оценка на test

In [ ]:
y_proba = cross_val_predict(
    best_clf, X_trainval, labels_trainval, cv=5, method="predict_proba"
)[:, config.CLASS_BAD]
threshold = find_optimal_threshold(labels_trainval, y_proba)

y_proba_test = best_clf.predict_proba(X_test)[:, config.CLASS_BAD]
test_metrics = evaluate(labels_test, y_proba_test, threshold=threshold, verbose=True)
pd.DataFrame({
    "path":    paths_test,
    "y_true":  labels_test,
    "y_pred":  (y_proba_test >= threshold).astype(int),
    "y_proba": y_proba_test,
}).to_csv(exp_dir / "test_predictions.csv", index=False)

save_result_csv(
    exp_dir=exp_dir,
    experiment_id="exp_wav2vec_svm",
    experiment_name="Wav2Vec2-base (frozen) + SVM",
    accuracy=test_metrics["accuracy"],
    f1_macro=test_metrics["f1_macro"],
    f1_bad=test_metrics["f1_bad"],
    roc_auc=test_metrics["roc_auc"],
    precision_bad=test_metrics["precision_bad"],
    recall_bad=test_metrics["recall_bad"],
    threshold=test_metrics["threshold"],
    embed_dim=2304,
    embed_dim_note="Wav2Vec2-base 768-dim × 3 (mean+std+max)",
    notes=f"GridSearchCV cv=5 + threshold | best_params={best_params} | thr={threshold:.2f}",
    train_time_sec=train_time_sec,
)